In [2]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random
import math
from sklearn.preprocessing import MinMaxScaler

In [3]:
data = pd.read_csv('../data/GOOG.csv')

In [4]:
features = data.drop(['Date', 'Volume'], axis=1)

In [5]:
class StockDataIter:
    def __init__(self, train_data, batch_size, num_steps, pre_steps):
        """
        参数说明:
        train_data: 缩放后的 NumPy 数组 (Total_Days, Features)
        batch_size: 每个批次的样本数
        num_steps: 输入序列长度 (过去多少天)
        pre_steps: 预测长度 (未来多少天)
        """
        self.train_data = train_data
        self.batch_size = batch_size
        self.num_steps = num_steps
        self.pre_steps = pre_steps

    def __iter__(self):
        # 1. 每一轮迭代时执行随机偏移，增加模型泛化能力
        offset = random.randint(0, self.num_steps - 1)
        data = self.train_data[offset:]
        
        # 2. 计算可产生的子序列数量 (确保不越界)
        num_subseqs = (len(data) - self.pre_steps) // self.num_steps
        
        # 3. 生成所有起始索引并打乱
        initial_indices = list(range(0, num_subseqs * self.num_steps, self.num_steps))
        random.shuffle(initial_indices)

        def get_x(pos):
            return data[pos : pos + self.num_steps]

        def get_y(pos):
            # 假设预测第 0 列 (Open)
            return data[pos : pos + self.pre_steps, 0]

        # 4. 生成批次
        num_batches = num_subseqs // self.batch_size
        for i in range(0, self.batch_size * num_batches, self.batch_size):
            batch_indices = initial_indices[i : i + self.batch_size]
            
            # 构造 X 和 Y
            X = np.array([get_x(j) for j in batch_indices])
            # Y 紧随 X 之后，起始位置为 j + num_steps
            Y = np.array([get_y(j + self.num_steps) for j in batch_indices])
            
            yield torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)

In [46]:
class RNNFromScratch(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNNFromScratch, self).__init__()
        self.hidden_size = hidden_size
        
        def three(input_size):
            return (nn.Parameter(torch.randn(input_size, hidden_size) * 0.01),
                   nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01),
                   nn.Parameter(torch.zeros(hidden_size)))

        # 参数
        self.W_xh, self.W_hh, self.b_h = three(input_size)  # 候选隐状态参数
        self.W_hh2, self.W_hh22, self.b_h2 = three(hidden_size)  # 候选隐状态参数
        
        # 输出层 (全连接层)：将最后的隐藏状态映射到未来7天的预测值
        self.W_hq = nn.Parameter(torch.randn(hidden_size, output_size) * 0.01)
        self.b_q = nn.Parameter(torch.zeros(output_size))

    def forward(self, inputs, H=None, H2=None):
        """
        inputs 形状: (batch_size, num_steps, input_size)
        state 形状:  (batch_size, hidden_size)
        """
        batch_size, num_steps, _ = inputs.shape
        
        # 如果没有初始状态，则初始化为全 0
        if H is None:
            H, H2 = torch.zeros((batch_size, self.hidden_size), device=inputs.device), torch.zeros((batch_size, self.hidden_size), device=inputs.device)
        
        # --- 2. 循环处理每一个时间步 ---
        # RNN 的本质：把序列拆开，一个一个时间点往后传
        for t in range(num_steps):
            X = inputs[:, t, :] # 获取当前时刻的输入 (batch_size, input_size)
            H = torch.sigmoid((X @ self.W_xh) + (H @ self.W_hh) + self.b_h)
            H2 = torch.sigmoid((H @ self.W_hh2) + (H2 @ self.W_hh22) + self.b_h2)
            
        # --- 3. 输出预测 ---
        # 使用最后一个时间步产生的隐藏状态来预测未来 7 天
        output = torch.matmul(H2, self.W_hq) + self.b_q
        return output

In [38]:
# class RNNFromScratch(nn.Module):
#     def __init__(self, input_size, hidden_size, output_size):
#         super(RNNFromScratch, self).__init__()
#         self.hidden_size = hidden_size
        
#         def three():
#             return (nn.Parameter(torch.randn(input_size, hidden_size) * 0.01),
#                    nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01),
#                    nn.Parameter(torch.zeros(hidden_size)))
    
#         # 参数
#         self.W_xi, self.W_hi, self.b_i = three()  # 输入门参数
#         self.W_xf, self.W_hf, self.b_f = three()  # 遗忘门参数
#         self.W_xo, self.W_ho, self.b_o = three()  # 输出门参数
#         self.W_xc, self.W_hc, self.b_c = three()  # 候选隐状态参数
        
#         # 输出层 (全连接层)：将最后的隐藏状态映射到未来7天的预测值
#         self.W_hq = nn.Parameter(torch.randn(hidden_size, output_size) * 0.01)
#         self.b_q = nn.Parameter(torch.zeros(output_size))
    
#     def forward(self, inputs, H=None, C=None):
#         """
#         inputs 形状: (batch_size, num_steps, input_size)
#         state 形状:  (batch_size, hidden_size)
#         """
#         batch_size, num_steps, _ = inputs.shape
        
#         # 如果没有初始状态，则初始化为全 0
#         if H is None:
#             H, C = torch.zeros((batch_size, self.hidden_size), device=inputs.device), torch.zeros((batch_size, self.hidden_size), device=inputs.device)
        
#         # --- 2. 循环处理每一个时间步 ---
#         # RNN 的本质：把序列拆开，一个一个时间点往后传
#         for t in range(num_steps):
#             X = inputs[:, t, :] # 获取当前时刻的输入 (batch_size, input_size)
    
#             I = torch.sigmoid((X @ self.W_xi) + (H @ self.W_hi) + self.b_i)
#             F = torch.sigmoid((X @ self.W_xf) + (H @ self.W_hf) + self.b_f)
#             O = torch.sigmoid((X @ self.W_xo) + (H @ self.W_ho) + self.b_o)
#             C_tilda = torch.tanh((X @ self.W_xc) + (H @ self.W_hc) + self.b_c)

#             C = F * C + I * C_tilda
#             H = O * torch.tanh(C)
        
#         # --- 3. 输出预测 ---
#         # 使用最后一个时间步产生的隐藏状态来预测未来 7 天
#         output = torch.matmul(H, self.W_hq) + self.b_q
#         return output

In [19]:
def train_rnn(model, data_iter, lr, epochs, device):
    """
    封装训练逻辑
    """
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.MSELoss()
    loss_history = []

    print("开始训练...")
    for epoch in range(epochs):
        epoch_loss = 0
        batch_count = 0
        
        for X, Y in data_iter:
            X, Y = X.to(device), Y.to(device)
            
            # 前向传播
            output = model(X)
            loss = criterion(output, Y)
            
            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            
            # 梯度裁剪：防止从零实现的 RNN 梯度爆炸
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            epoch_loss += loss.item()
            batch_count += 1
            
        avg_loss = epoch_loss / batch_count
        loss_history.append(avg_loss)
        
        if (epoch + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.6f}")
            
    return loss_history

In [20]:
def predict_next_7_days(model, full_data, num_steps, device):
    """
    封装预测逻辑 (针对未缩放的数据)
    参数:
    model: 训练好的模型
    full_data: 原始数据矩阵 (N, 特征数)，通常是 features.values
    num_steps: 观察窗口大小 (如 30)
    device: 设备 (cpu 或 cuda)
    """
    model.eval()
    model.to(device)
    
    # 1. 提取最后 num_steps 天的数据作为输入
    # 直接取原始数值，不需要缩放
    last_window = full_data[-num_steps:] 
    
    # 2. 转换为张量，形状为 (1, num_steps, 特征数)
    input_tensor = torch.tensor(last_window, dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # 3. 得到未来 7 天的预测值
        # 输出形状通常是 [1, 7]，我们通过 flatten() 转为一维数组
        predictions = model(input_tensor).cpu().numpy().flatten()
    
    # 因为没有进行过缩放，这里的 predictions 直接就是原始价格数值
    return predictions

In [47]:
# --- 配置参数 ---
INPUT_SIZE = 5    # 特征数 (Open, High, Low, Close, Adj Close, Volume)
HIDDEN_SIZE = 128
OUTPUT_SIZE = 7   # 预测未来7天
BATCH_SIZE = 32
NUM_STEPS = 30    # 观察过去30天
LR = 0.001
EPOCHS = 500
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_iter = StockDataIter(features.values, BATCH_SIZE, NUM_STEPS, OUTPUT_SIZE)

model = RNNFromScratch(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)


In [48]:
losses = train_rnn(model, data_iter, LR, EPOCHS, DEVICE)

开始训练...
Epoch [50/500], Loss: 1535.146948
Epoch [100/500], Loss: 569.284497
Epoch [150/500], Loss: 212.378207
Epoch [200/500], Loss: 82.417229
Epoch [250/500], Loss: 44.877278
Epoch [300/500], Loss: 26.220389
Epoch [350/500], Loss: 21.736968
Epoch [400/500], Loss: 12.972639
Epoch [450/500], Loss: 7.864326
Epoch [500/500], Loss: 6.295562
